# Brain Tumour Classifier — ResNet Transfer Learning

Three-class brain-tumour MRI classification (**Meningioma, Glioma, Pituitary**) using
**ResNet-50** transfer learning with 8x rotation augmentation.

Reference implementation for the paper *"Classification Of Brain Tumor Using Resnet50"*
(Anand A., Ridhuparan K., Karthik G.S., Sooraj Veer R., Raghu Prasath V. —
*Solid State Technology*, Vol. 63 No. 5, 2020).

> Originally written for Google Colab; cleaned up here to run in any local Jupyter
> environment. See the repository `README.md` for setup and dataset instructions.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import os
import random
import numpy as np
import pandas as pd
import pickle
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


## Configuration

Edit these paths for your environment. See `data/README.md` for the expected dataset format.

In [ ]:
# Path to the pickled dataset: a list of (image, label) pairs.
# Labels are 1..3 -> Meningioma, Glioma, Pituitary. See data/README.md.
DATA_PATH = os.path.join("..", "data", "training_data.pickle")

# Where model checkpoints / weights are written.
CKPT_DIR = os.path.join("..", "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)


In [ ]:
torch.__version__

In [ ]:
torch.cuda.empty_cache()


## Dataset

Each sample is returned as its original tensor plus 7 rotation-augmented copies (8x augmentation).

In [ ]:
class BrainTumorDataset(Dataset):
  def __init__(self, images, labels):
    # images
    self.X = images
    # labels
    self.y = labels

    # convert it to a tensor
    self.transform = transforms.Compose([transforms.ToPILImage(),
        transforms.ToTensor()
    ])

    # between -45 degrees and 45 degrees, tensor
    self.transform1 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(30),
        transforms.ToTensor()
    ])

    #  -90 degrees and 90 degrees, tensor
    self.transform2 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(60),
        transforms.ToTensor()
    ])

    # -120 degrees and 120 degrees,tensor
    self.transform3 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(90),
        transforms.ToTensor()
    ])

    #-180 degrees and 180 degrees, tensor
    self.transform4 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(135),
        transforms.ToTensor()
    ])

    # -270 degrees and 270 degrees, tensor
    self.transform5 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(180),
        transforms.ToTensor()
    ])

    # -300 degrees and 300 degrees, tensor
    self.transform6 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(225),
        transforms.ToTensor()
    ])

    # -330 degrees and 330 degrees, tensor
    self.transform7 = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomRotation(270),
        transforms.ToTensor()
    ])

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    # perform transformations on one instance of X
    # Original image as a tensor
    data = self.transform(self.X[idx])


    aug30 = self.transform1(self.X[idx])


    aug60 = self.transform2(self.X[idx])


    aug90 = self.transform3(self.X[idx])


    aug135 = self.transform4(self.X[idx])


    aug180 = self.transform5(self.X[idx])


    aug225 = self.transform6(self.X[idx])


    aug270 = self.transform7(self.X[idx])


    new_batch = [data, aug30, aug60, aug90, aug135, aug180, aug225, aug270]


    # store the network's understandable label as a tensor
    labels = torch.tensor((self.y[idx]-1))


    return (labels, new_batch)

## Load and split the data (70% train / 15% val / 15% test)

In [ ]:
training_data = pickle.load(open(DATA_PATH, "rb"))


In [ ]:

Xt = []
yt = []
features = None
labels = None
label = []

In [ ]:
for features,labels in training_data:
  Xt.append(features)
  yt.append(labels)

In [ ]:
# 70% training, 15% validation, 15% testing
random.seed(51)

# First split off 30% for validation + testing
X_train, X_rem, y_train, y_rem = train_test_split(
    Xt, yt, test_size=0.3, shuffle=True, random_state=33)

# Split the held-out 30% evenly into validation (15%) and testing (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_rem, y_rem, test_size=0.5, shuffle=True, random_state=33)


In [ ]:
train_set = BrainTumorDataset(X_train, y_train)
valid_set = BrainTumorDataset(X_valid, y_valid)
test_set = BrainTumorDataset(X_test, y_test)

In [ ]:

print(f"Number of training samples: {len(X_train)}")
print(f"Number of validation samples: {len(X_valid)}")
print(f"Number of testing samples: {len(X_test)}")

In [ ]:
print(f"Number of augmented training samples: {len(X_train) * 8}")
print(f"Number of augmented validation samples: {len(X_valid)* 8}")
print(f"Number of augmented testing samples: {len(X_test)* 8}")

In [ ]:
train_gen = DataLoader(train_set, batch_size=4, shuffle=True, pin_memory=True, num_workers=8)
valid_gen = DataLoader(valid_set, batch_size=4, shuffle=True, pin_memory=True, num_workers=8)
test_gen = DataLoader(test_set, batch_size=10, shuffle=True, pin_memory=True, num_workers=8)

## Device

In [ ]:
torch.cuda.current_device()
torch.cuda.device(0)
torch.cuda.get_device_name(0)

In [ ]:
device_name = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_name)

## Model — ResNet-50 with a custom classification head

In [ ]:

# instantiate transfer learning model
resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# set all paramters as trainable
for param in resnet_model.parameters():
    param.requires_grad = True

# get input of fc layer
n_inputs = resnet_model.fc.in_features

# redefine fc layer / top layer/ head for our classification problem
resnet_model.fc = nn.Sequential(nn.Linear(n_inputs, 2048),
                                nn.LeakyReLU(negative_slope=0.2),
                                nn.Dropout(p=0.4),
                                nn.Linear(2048, 2048),
                                nn.LeakyReLU(negative_slope=0.2),
                                nn.Dropout(p=0.4),
                                nn.Linear(2048, 3),
                                nn.LogSoftmax(dim=1))

# set all paramters of the model as trainable
for name, child in resnet_model.named_children():
  for name2, params in child.named_parameters():
    params.requires_grad = True

# set model to run on GPU or CPU absed on availibility
resnet_model.to(device)

# print the trasnfer learning NN model's architecture
resnet_model

## Loss, optimizer, and training hyperparameters

In [ ]:
# The classifier head ends in LogSoftmax, so NLLLoss is the matching loss for
# single-label 3-class classification.
criterion = nn.NLLLoss()
if device_name == "cuda":
    criterion = criterion.cuda()

optimizer = torch.optim.SGD(resnet_model.parameters(), momentum=0.9, lr=0.0003)

epochs = 10

# metric history for plotting
train_losses = []
val_losses = []
train_accs = []
val_accs = []


In [ ]:
def save_checkpoint(state, is_best, filename=None):
    if filename is None:
        filename = os.path.join(CKPT_DIR, "bt_total_resnet_checkpoint.pth.tar")
    torch.save(state, filename)


### Un-batching helper

The `DataLoader` yields `y` of shape `[B]` and `X` as a **list of 8 tensors** (one per
augmentation), each `[B, 3, 512, 512]`. This helper stacks them into a single
`[B*8, 3, 512, 512]` batch and repeats each label 8x so every augmented image keeps its
class label.

In [ ]:
def flatten_batch(y, X):
    # X: list of 8 tensors [B, 3, 512, 512] -> images [B*8, 3, 512, 512]
    images = torch.stack(X, dim=1).reshape(-1, 3, 512, 512)
    # one label per augmented image, aligned with the stack order
    labels = y.repeat_interleave(len(X))
    return images, labels


## Training

In [ ]:
start_time = time.time()
best_val_loss = float("inf")

for epoch in range(epochs):
    # ---- train ----
    resnet_model.train()
    trn_corr = trn_total = 0
    e_start = time.time()

    for b, (y, X) in enumerate(train_gen):
        images, labels = flatten_batch(y, X)
        images, labels = images.to(device), labels.to(device)

        y_pred = resnet_model(images)
        loss = criterion(y_pred, labels)

        predicted = torch.argmax(y_pred, dim=1)
        trn_corr += (predicted == labels).sum().item()
        trn_total += labels.size(0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if device_name == "cuda":
            torch.cuda.empty_cache()

    train_acc = 100 * trn_corr / trn_total
    train_losses.append(loss.item())
    train_accs.append(train_acc)
    print(f"Epoch {epoch+1}/{epochs}  "
          f"Train Acc: {train_acc:5.2f}%  Loss: {loss.item():.4f}  "
          f"Duration: {(time.time()-e_start)/60:.2f} min")

    # ---- validate ----
    resnet_model.eval()
    val_corr = val_total = 0
    with torch.no_grad():
        for y, X in valid_gen:
            images, labels = flatten_batch(y, X)
            images, labels = images.to(device), labels.to(device)

            y_val = resnet_model(images)
            val_loss = criterion(y_val, labels)

            predicted = torch.argmax(y_val, dim=1)
            val_corr += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_corr / val_total
    val_losses.append(val_loss.item())
    val_accs.append(val_acc)
    print(f"          Val   Acc: {val_acc:5.2f}%  Loss: {val_loss.item():.4f}\n")

    # checkpoint on best (lowest) validation loss
    is_best = val_loss.item() < best_val_loss
    best_val_loss = min(val_loss.item(), best_val_loss)
    save_checkpoint({
        "epoch": epoch + 1,
        "state_dict": resnet_model.state_dict(),
        "best_val_loss": best_val_loss,
    }, is_best)

print("\nTraining Duration {:.2f} minutes".format((time.time()-start_time)/60))
if device_name == "cuda":
    print("GPU memory used    : {} bytes".format(torch.cuda.memory_allocated()))
    print("GPU memory reserved: {} bytes".format(torch.cuda.memory_reserved()))


In [ ]:
torch.save(resnet_model.state_dict(), os.path.join(CKPT_DIR, "bt_total_resnet_torch.pt"))


In [ ]:
print(f"Final validation accuracy: {val_accs[-1]:.2f}%")


## Training curves

In [ ]:
plt.plot(train_losses, label='Training loss')
plt.plot(val_losses, label='Validation loss')
plt.title('Loss Metrics')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
plt.show()


In [ ]:
plt.plot(train_accs, label='Training accuracy')
plt.plot(val_accs, label='Validation accuracy')
plt.title('Accuracy Metrics')
plt.ylabel('Accuracy (%)')
plt.xlabel('Epochs')
plt.legend()
plt.show()


## Evaluation on the test set

In [ ]:
# Free training generators before evaluation
# To reload weights instead of retraining:
# resnet_model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "bt_total_resnet_torch.pt")))
train_gen = None
valid_gen = None
train_set = None
valid_set = None


In [ ]:
resnet_model.eval()

test_corr = test_total = 0
test_losses_list = []
all_labels = []
all_preds = []

with torch.no_grad():
    for y, X in test_gen:
        images, labels = flatten_batch(y, X)
        images, labels = images.to(device), labels.to(device)

        y_val = resnet_model(images)
        loss = criterion(y_val, labels)

        predicted = torch.argmax(y_val, dim=1)
        test_corr += (predicted == labels).sum().item()
        test_total += labels.size(0)
        test_losses_list.append(loss.item())

        all_labels.append(labels.cpu())
        all_preds.append(predicted.cpu())

print(f"Test Loss: {test_losses_list[-1]:.4f}")


In [ ]:
print(f"Test accuracy: {100 * test_corr / test_total:.2f}%")


In [ ]:
# concatenate per-batch predictions/targets into flat 1-D tensors
all_labels = torch.cat(all_labels)
all_preds = torch.cat(all_preds)


## Results — confusion matrix and classification report

In [ ]:
LABELS = ['Meningioma', 'Glioma', 'Pitutary']

In [ ]:
arr = confusion_matrix(all_labels.numpy(), all_preds.numpy())
df_cm = pd.DataFrame(arr, index=LABELS, columns=LABELS)
plt.figure(figsize=(9, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap='viridis')
plt.xlabel("Prediction")
plt.ylabel("Target")
plt.show()


In [ ]:
print("Classification Report\n")
print(classification_report(all_labels.numpy(), all_preds.numpy(), target_names=LABELS))
